# 第十二课：NumPy 位运算

本课依据菜鸟教程的 [NumPy 位运算](https://www.runoob.com/numpy/numpy-binary-operators.html) 整理，学习按位与、或、异或、取反、移位，以及位掩码的实际应用。

位运算直接操作整数的二进制位，常见于权限标志、颜色通道、压缩数据和底层协议。

In [1]:
import numpy as np
print("NumPy 版本：", np.__version__)

NumPy 版本： 2.5.3


## 1. 查看二进制表示

Python 的 `bin()` 适合单个整数；`np.binary_repr(value, width)` 可以指定显示位宽。

In [14]:
for value in [5, 10, 13, 17]:
    print(value, "→", np.binary_repr(value, width=8),bin(value))

5 → 00000101 0b101
10 → 00001010 0b1010
13 → 00001101 0b1101
17 → 00010001 0b10001


## 2. 按位与 AND：`&` / `bitwise_and()`

只有两个对应位都是 1，结果位才是 1。常用于检查某些标志位是否开启。

In [ ]:
a = np.array([13, 7], dtype=np.uint8)
b = np.array([17, 3], dtype=np.uint8)
result = np.bitwise_and(a, b)
print("a：", [np.binary_repr(x, width=8) for x in a])
print("b：", [np.binary_repr(x, width=8) for x in b])
print("a & b：", result)#这里按位与的意思是两个向量二进制重合的部分的值，比如这里是00000001，00000011，分别对应1和3所以结果就算[1 3]
print("等价运算符：", a & b)

a： ['00001101', '00000111']
b： ['00010001', '00000011']
a & b： [1 3]
等价运算符： [1 3]


## 3. 按位或 OR 与异或 XOR

- OR：只要有一个位是 1，结果就是 1，常用于开启标志位。
- XOR：两个位不同时结果为 1，常用于切换标志位或寻找差异。

In [ ]:
x = np.array([10], dtype=np.uint8)  # 00001010
y = np.array([12], dtype=np.uint8)  # 00001100
for name, value in [("x | y", x | y), ("x ^ y", x ^ y)]:#前面是按位或，后面是异或
    print(name, "=", value[0], "→", np.binary_repr(value[0], width=8))

x | y = 14 → 00001110
x ^ y = 6 → 00000110


## 4. 按位取反：`~` / `invert()`

取反会翻转固定宽度整数中的每一位。结果取决于 `dtype`：`uint8` 的 8 位都被翻转；有符号整数使用补码表示，常见关系是 `~x == -x - 1`。

In [5]:
unsigned = np.array([5], dtype=np.uint8)
signed = np.array([5], dtype=np.int8)
print("uint8 5：", np.binary_repr(unsigned[0], width=8), "取反后：", (~unsigned)[0], np.binary_repr((~unsigned)[0], width=8))
print("int8 5 取反后：", (~signed)[0])
print("验证 ~x == -x-1：", (~signed)[0] == -signed[0] - 1)

uint8 5： 00000101 取反后： 250 11111010
int8 5 取反后： -6
验证 ~x == -x-1： True


## 5. 左移与右移

对非负整数，在不溢出的前提下：左移 `n` 位相当于乘以 $2^n$，右移 `n` 位相当于整除 $2^n$。固定宽度类型左移过多可能溢出。

In [16]:
values = np.array([5, 10, 20], dtype=np.uint16)
print("原值：", values)
print("左移 2 位：", np.left_shift(values, 2))
print("右移 1 位：", np.right_shift(values, 1))
print("运算符写法：", values << 2, values >> 1)

原值： [ 5 10 20]
左移 2 位： [20 40 80]
右移 1 位： [ 2  5 10]
运算符写法： [20 40 80] [ 2  5 10]


## 6. 布尔数组上的位运算

对布尔数组，`&`、`|`、`^`、`~` 分别执行逐元素逻辑与、或、异或、非。这也是组合 NumPy 筛选条件时使用它们的原因。注意不要用 Python 的 `and`、`or` 处理整个数组。

In [7]:
p = np.array([True, True, False, False])
q = np.array([True, False, True, False])
print("p & q：", p & q)
print("p | q：", p | q)
print("p ^ q：", p ^ q)
print("~p：", ~p)

p & q： [ True False False False]
p | q： [ True  True  True False]
p ^ q： [False  True  True False]
~p： [False False  True  True]


## 7. 实际应用：权限位掩码

假设最低三位分别表示读取、写入、执行权限：

- `READ = 001`
- `WRITE = 010`
- `EXECUTE = 100`

用 OR 组合权限，用 AND 检查权限，用 XOR 切换权限，用 `& ~MASK` 关闭权限。

In [20]:
READ = np.uint8(0b001)
WRITE = np.uint8(0b010)
EXECUTE = np.uint8(0b100)
permissions = np.array([0b001, 0b011, 0b101, 0b111], dtype=np.uint8)
print("权限值：", permissions)
print("可以写入：", (permissions & WRITE) != 0)
print("同时可读可执行：", (permissions & (READ | EXECUTE)) == (READ | EXECUTE))#[] $101 == 101
enabled_write = permissions | WRITE
disabled_execute = permissions & np.invert(EXECUTE)
print("全部开启写入：", enabled_write)
print("全部关闭执行：", disabled_execute)

权限值： [1 3 5 7]
可以写入： [False  True False  True]
同时可读可执行： [False False  True  True]
全部开启写入： [3 3 7 7]
全部关闭执行： [1 3 1 3]


## 8. `packbits()` 与 `unpackbits()`

`packbits()` 把 0/1 数组压缩成字节，8 个布尔位可放进 1 个 `uint8`。`unpackbits()` 执行反向操作。适合大量二值标记。

In [ ]:
bits = np.array([1, 0, 1, 1, 0, 0, 1, 0], dtype=np.uint8)
packed = np.packbits(bits)#为二进制表示的十进制数的列表
restored = np.unpackbits(packed)
print("位：", bits)
print("压缩字节：", packed, "二进制：", np.binary_repr(packed[0], width=8))
print("还原：", restored)
print("压缩前/后字节数：", bits.nbytes, "/", packed.nbytes)

位： [1 0 1 1 0 0 1 0]
压缩字节： [178] 二进制： 10110010
还原： [1 0 1 1 0 0 1 0]
压缩前/后字节数： 8 / 1


## 9. 位运算同样支持广播

下面使用同一个掩码检查数组中每个整数的最低位。最低位为 1 就是奇数。

In [ ]:
numbers = np.arange(10, dtype=np.uint8)
least_significant_bit = numbers & np.uint8(1)#检查二进制末位是不是1，是1就算奇数
print("数字：", numbers)
print("最低位：", least_significant_bit)
print("奇数：", numbers[least_significant_bit == 1])

数字： [0 1 2 3 4 5 6 7 8 9]
最低位： [0 1 0 1 0 1 0 1 0 1]
奇数： [1 3 5 7 9]


## 10. 常见陷阱

- 位运算主要用于整数和布尔类型，不适用于普通浮点数。
- 固定宽度整数可能在左移时溢出。
- `~` 的结果取决于有符号/无符号类型及位宽。
- 比较条件与位运算混用时要加括号，例如 `(x > 2) & (x < 8)`。
- `&` 是逐元素位运算，`and` 是 Python 单一真值判断，两者不可混用。

In [11]:
small = np.array([128], dtype=np.uint8)
print("128 的二进制：", np.binary_repr(small[0], width=8))
print("uint8 左移一位发生溢出：", small << 1)
print("先转为 uint16：", small.astype(np.uint16) << 1)

128 的二进制： 10000000
uint8 左移一位发生溢出： [0]
先转为 uint16： [256]


## 本课小结

- `&`、`|`、`^`、`~` 分别是与、或、异或、取反。
- `<<`、`>>` 分别是左移和右移。
- NumPy 位运算逐元素执行并支持广播。
- 位掩码适合组合、检查、开启和关闭标志。
- `packbits()` 与 `unpackbits()` 可压缩和还原二值数据。
- 固定宽度整数要注意位宽、有无符号和溢出。

## 本节练习

定义状态位：在线 `ONLINE=001`、管理员 `ADMIN=010`、已验证 `VERIFIED=100`。对 `users` 完成：

1. 筛选在线用户。
2. 筛选同时是管理员且已验证的用户。
3. 为所有用户开启已验证状态。
4. 为所有用户关闭管理员状态。
5. 用 `binary_repr(..., width=3)` 输出修改前后的二进制状态。

In [33]:
ONLINE = np.uint8(0b001)
ADMIN = np.uint8(0b010)
VERIFIED = np.uint8(0b100)
users = np.array([0b001, 0b011, 0b100, 0b110, 0b111], dtype=np.uint8)
# 请在这里完成练习
print(users)
print("online",users[users & ONLINE != 0])
print('admin and verified',users[(users & (ADMIN | VERIFIED)) == 6])
users_ = users | VERIFIED 
print(users_)
users__ = users & ~ADMIN
print(users__)
print([np.binary_repr(x,width=3) for x in users])
print([np.binary_repr(x,width=3) for x in users_])
print([np.binary_repr(x,width=3) for x in users__])
 




[1 3 4 6 7]
online [1 3 7]
admin and verified [6 7]
[5 7 4 6 7]
[1 1 4 4 5]
['001', '011', '100', '110', '111']
['101', '111', '100', '110', '111']
['001', '001', '100', '100', '101']


## 参考答案

In [13]:
online_users = users[(users & ONLINE) != 0]
admin_verified = users[(users & (ADMIN | VERIFIED)) == (ADMIN | VERIFIED)]
all_verified = users | VERIFIED
no_admin = users & np.invert(ADMIN)
print("在线用户状态：", online_users)
print("管理员且已验证：", admin_verified)
print("全部验证：", all_verified)
print("关闭管理员：", no_admin)
print("原状态：", [np.binary_repr(x, width=3) for x in users])
print("全部验证后：", [np.binary_repr(x, width=3) for x in all_verified])
print("关闭管理员后：", [np.binary_repr(x, width=3) for x in no_admin])

在线用户状态： [1 3 7]
管理员且已验证： [6 7]
全部验证： [5 7 4 6 7]
关闭管理员： [1 1 4 4 5]
原状态： ['001', '011', '100', '110', '111']
全部验证后： ['101', '111', '100', '110', '111']
关闭管理员后： ['001', '001', '100', '100', '101']


---

下一课建议学习：[NumPy 字符串函数](https://www.runoob.com/numpy/numpy-string-functions.html)。